<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 5


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Book в C#, который будет представлять информацию о книгах. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// ДЕЛЕГАТЫ
public delegate void BookActionDelegate(Book book);
public delegate bool BookFilterDelegate(Book book);
public delegate string BookFormatDelegate(Book book);
public delegate void LibraryEventHandler(string message, Book book);

// КЛАСС READER
public class Reader
{
    private string _name;
    private string _email;
    private List<Book> _borrowedBooks;
    private int _maxBooks;

    // СОБЫТИЕ
    public event EventHandler<BookBorrowedEventArgs> BookBorrowed;
    
    public Reader(string name, string email, int maxBooks = 10)
    {
        Name = name;
        Email = email;
        _maxBooks = maxBooks;
        _borrowedBooks = new List<Book>();
        BorrowHistory = new List<BorrowRecord>();
    }

    public string Name
    {
        get { return _name; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _name = value;
            else
                throw new ArgumentException("Имя читателя не может быть пустым!");
        }
    }

    public string Email
    {
        get { return _email; }
        set 
        { 
            if (!string.IsNullOrEmpty(value) && value.Contains("@"))
                _email = value;
            else
                throw new ArgumentException("Email должен быть корректным!");
        }
    }

    // НОВАЯ КОЛЛЕКЦИЯ
    public List<BorrowRecord> BorrowHistory { get; private set; }

    public void AddBorrowedBook(Book book)
    {
        if (_borrowedBooks.Count < _maxBooks)
        {
            _borrowedBooks.Add(book);
            
            // Добавляем запись в историю
            BorrowHistory.Add(new BorrowRecord(book, DateTime.Now));
            
            // Вызываем событие
            OnBookBorrowed(book);
        }
        else
        {
            Console.WriteLine($"{Name} уже взял максимальное количество книг");
        }
    }

    public void ProcessBorrowedBooks(BookActionDelegate processor)
    {
        Console.WriteLine($"\nОбработка книг читателя {Name}:");
        foreach (var book in _borrowedBooks)
        {
            processor(book);
        }
    }

    public IEnumerable<Book> FilterBorrowedBooks(Func<Book, bool> filter)
    {
        return _borrowedBooks.Where(filter);
    }

    public void RemoveBorrowedBook(Book book)
    {
        if (_borrowedBooks.Remove(book))
        {
            var record = BorrowHistory.FirstOrDefault(r => r.Book == book && !r.Returned);
            if (record != null)
            {
                record.MarkReturned();
            }
        }
    }

    public void DisplayBorrowedBooks()
    {
        Console.WriteLine($"\nКниги, взятые {Name}:");
        
        if (!_borrowedBooks.Any())
        {
            Console.WriteLine("Нет взятых книг");
            return;
        }

        _borrowedBooks.ForEach(book => 
            Console.WriteLine($"- {book.Title} ({book.GetType().Name})"));
    }

    public void DisplayBorrowHistory()
    {
        Console.WriteLine($"\nИстория заимствований {Name}:");
        if (!BorrowHistory.Any())
        {
            Console.WriteLine("Нет записей в истории");
            return;
        }

        BorrowHistory.ForEach(record => Console.WriteLine(record));
    }

    protected virtual void OnBookBorrowed(Book book)
    {
        BookBorrowed?.Invoke(this, new BookBorrowedEventArgs(book));
    }
}


public class BorrowRecord
{
    public Book Book { get; }
    public DateTime BorrowDate { get; }
    public DateTime? ReturnDate { get; private set; }
    public bool Returned => ReturnDate.HasValue;

    public BorrowRecord(Book book, DateTime borrowDate)
    {
        Book = book;
        BorrowDate = borrowDate;
    }

    public void MarkReturned()
    {
        ReturnDate = DateTime.Now;
    }

    public override string ToString()
    {
        return $"{Book.Title} - Взята: {BorrowDate:dd.MM.yyyy}, " +
               $"{(Returned ? $"Возвращена: {ReturnDate:dd.MM.yyyy}" : "Еще не возвращена")}";
    }
}


public class BookBorrowedEventArgs : EventArgs
{
    public Book Book { get; }
    
    public BookBorrowedEventArgs(Book book)
    {
        Book = book;
    }
}

public interface ILibraryItem
{
    string Title { get; }
    string Author { get; }
    void DisplayInfo();
    decimal CalculateValue();
}

public interface IBorrowable
{
    bool IsAvailable { get; }
    void Borrow(Reader reader);
    void Return();
}

public interface IResearchable : ILibraryItem
{
    void Research();
    int GetCitationCount();
}

// БАЗОВЫЙ КЛАСС BOOK 
public class Book : ILibraryItem, IBorrowable
{
    private static int _nextId = 1;
    
    private string _title;
    private string _author;
    private int _yearOfPublication;
    private bool _isBorrowed;
    private Reader _currentReader;
    private int _pageCount;
    private string _publisher;
    private decimal _price;
    private int _readCount;
    private List<string> _tags;

    
    public event EventHandler<BookStatusChangedEventArgs> StatusChanged;
    public event LibraryEventHandler BookRead;

    public Book(string title, string author, int yearOfPublication, int pageCount, 
                string publisher, decimal price)
    {
        Id = _nextId++;
        Title = title;
        Author = author;
        YearOfPublication = yearOfPublication;
        PageCount = pageCount;
        Publisher = publisher;
        Price = price;
        IsBorrowed = false;
        CurrentReader = null;
        _readCount = 0;
        _tags = new List<string>();
        CreationDate = DateTime.Now;
        Condition = 100;
    }

    // НОВЫЕ АТРИБУТЫ
    public int Id { get; private set; }
    public DateTime CreationDate { get; private set; }
    
    public int Condition { get; protected set; }

    // СВОЙСТВА
    public string Title
    {
        get { return _title; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _title = value;
            else
                throw new ArgumentException("Название книги не может быть пустым!");
        }
    }

    public string Author
    {
        get { return _author; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _author = value;
            else
                throw new ArgumentException("Автор не может быть пустым!");
        }
    }

    public int YearOfPublication
    {
        get { return _yearOfPublication; }
        set 
        { 
            if (value > 0 && value <= DateTime.Now.Year)
                _yearOfPublication = value;
            else
                throw new ArgumentOutOfRangeException("Год издания должен быть положительным и не больше текущего года!");
        }
    }

  
    public int PageCount
    {
        get { return _pageCount; }
        set 
        { 
            if (value >= 0)  // Изменено с value > 0 на value >= 0
                _pageCount = value;
            else
                throw new ArgumentException("Количество страниц не может быть отрицательным!");
        }
    }

    public string Publisher
    {
        get { return _publisher; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _publisher = value;
            else
                throw new ArgumentException("Издатель не может быть пустым!");
        }
    }

    public decimal Price
    {
        get { return _price; }
        set 
        { 
            if (value >= 0)
                _price = value;
            else
                throw new ArgumentException("Цена не может быть отрицательной!");
        }
    }

    
    public int ReadCount
    {
        get { return _readCount; }
        internal set { _readCount = value; }
    }

    public bool IsBorrowed
    {
        get { return _isBorrowed; }
        private set { _isBorrowed = value; }
    }

    public Reader CurrentReader
    {
        get { return _currentReader; }
        private set { _currentReader = value; }
    }

    
    public IReadOnlyList<string> Tags => _tags.AsReadOnly();

    
    public void AddTag(string tag)
    {
        if (!string.IsNullOrEmpty(tag) && !_tags.Contains(tag))
        {
            _tags.Add(tag);
            Console.WriteLine($"Добавлен тег '{tag}' к книге '{Title}'");
        }
    }

    
    public bool HasTag(Predicate<string> tagMatcher)
    {
        return _tags.Exists(tagMatcher);
    }

    
    string ILibraryItem.Title => Title;
    string ILibraryItem.Author => Author;

    void ILibraryItem.DisplayInfo()
    {
        Console.WriteLine(GetInfo());
    }

    //публичный метод CalculateValue
    public virtual decimal CalculateValue()
    {
        decimal baseValue = Price;
        if (YearOfPublication < 1950) baseValue *= 2m;
        if (ReadCount > 100) baseValue *= 1.5m;
        baseValue *= (Condition / 100m);
        return baseValue;
    }

    // ДОБАВЛЕНО: реализация для интерфейса
    decimal ILibraryItem.CalculateValue() => CalculateValue();

    
    bool IBorrowable.IsAvailable => !IsBorrowed;

    void IBorrowable.Borrow(Reader reader)
    {
        Borrow(reader);
    }

    void IBorrowable.Return()
    {
        Return();
    }

    // СУЩЕСТВУЮЩИЕ И НОВЫЕ МЕТОДЫ
    public virtual void Read()
    {
        ReadCount++;
        Condition = Math.Max(0, Condition - 1);
        
        OnBookRead($"Прочитана книга '{Title}'", this);
        
        Console.WriteLine($"Чтение книги: '{Title}' (страниц: {PageCount})");
        Console.WriteLine($"Состояние книги: {Condition}%");
    }

    public virtual void Read(int pages)
    {
        ReadCount++;
        Condition = Math.Max(0, Condition - pages / 10);
        
        OnBookRead($"Прочитано {pages} страниц книги '{Title}'", this);
        
        Console.WriteLine($"Чтение книги: '{Title}' - прочитано {pages} страниц из {PageCount}");
        Console.WriteLine($"Состояние книги: {Condition}%");
    }

    public virtual void Restore(int conditionPoints)
    {
        Condition = Math.Min(100, Condition + conditionPoints);
        Console.WriteLine($"Книга '{Title}' отреставрирована. Текущее состояние: {Condition}%");
    }

    public virtual string GetInfo()
    {
        string status = IsBorrowed ? $" (Взята читателем: {CurrentReader?.Name})" : " (Доступна)";
        string tagsInfo = _tags.Any() ? $", Теги: {string.Join(", ", _tags)}" : "";
        return $"Книга ID:{Id}: '{Title}', Автор: {Author}, Год: {YearOfPublication}, " +
               $"Страниц: {PageCount}, Издатель: {Publisher}, Состояние: {Condition}%{tagsInfo}{status}";
    }

    public virtual void Borrow(Reader reader)
    {
        if (IsBorrowed)
        {
            Console.WriteLine($"Книга '{Title}' уже взята читателем {CurrentReader.Name}");
            return;
        }

        IsBorrowed = true;
        CurrentReader = reader;
        
        OnStatusChanged("Выдана", this);
        
        Console.WriteLine($"Книга '{Title}' выдана читателю {reader.Name}");
        reader.AddBorrowedBook(this);
    }

    public void Return()
    {
        if (!IsBorrowed)
        {
            Console.WriteLine($"Книга '{Title}' уже возвращена");
            return;
        }

        Console.WriteLine($"Книга '{Title}' возвращена читателем {CurrentReader.Name}");
        IsBorrowed = false;
        
        OnStatusChanged("Возвращена", this);
        
        CurrentReader?.RemoveBorrowedBook(this);
        CurrentReader = null;
    }

    public bool IsAvailable => !IsBorrowed;

    // НОВЫЙ МЕТОД для форматирования информации
    public string FormatInfo(BookFormatDelegate formatter)
    {
        return formatter(this);
    }

    protected virtual void OnStatusChanged(string action, Book book)
    {
        StatusChanged?.Invoke(this, new BookStatusChangedEventArgs(action, book));
    }

    protected virtual void OnBookRead(string message, Book book)
    {
        BookRead?.Invoke(message, book);
    }
}

// НОВЫЙ КЛАСС для аргументов события изменения статуса
public class BookStatusChangedEventArgs : EventArgs
{
    public string Action { get; }
    public Book Book { get; }
    
    public BookStatusChangedEventArgs(string action, Book book)
    {
        Action = action;
        Book = book;
    }
}

// НОВЫЙ КЛАСС для управления библиотекой
public class Library
{
    private Dictionary<int, Book> _books;
    private List<Reader> _readers;
    
    
    public event EventHandler<BookAddedEventArgs> BookAdded;
    public event EventHandler<LibraryStatisticsEventArgs> StatisticsUpdated;
    
    public Library()
    {
        _books = new Dictionary<int, Book>();
        _readers = new List<Reader>();
    }
    
    public void AddBook(Book book)
    {
        _books[book.Id] = book;
        OnBookAdded(book);
        UpdateStatistics();
    }
    
    public void AddReader(Reader reader)
    {
        _readers.Add(reader);
        UpdateStatistics();
    }
    
    public Book FindBookById(int id)
    {
        return _books.TryGetValue(id, out var book) ? book : null;
    }
    
    public IEnumerable<Book> FindBooks(BookFilterDelegate filter)
    {
        return _books.Values.Where(book => filter(book));
    }
    
    public IEnumerable<Book> GetBooksByAuthor(string author)
    {
        return _books.Values.Where(b => 
            b.Author.Equals(author, StringComparison.OrdinalIgnoreCase));
    }
    
    public void ProcessAllBooks(BookActionDelegate action)
    {
        Console.WriteLine($"\nОбработка всех книг в библиотеке ({_books.Count} шт.):");
        foreach (var book in _books.Values)
        {
            action(book);
        }
    }
    
    public void DisplayStatistics()
    {
        int availableBooks = _books.Values.Count(b => b.IsAvailable);
        int borrowedBooks = _books.Values.Count(b => !b.IsAvailable);
        decimal totalValue = _books.Values.Sum(b => b.CalculateValue());
        
        Console.WriteLine("\n=== СТАТИСТИКА БИБЛИОТЕКИ ===");
        Console.WriteLine($"Всего книг: {_books.Count}");
        Console.WriteLine($"Доступно: {availableBooks}");
        Console.WriteLine($"Выдано: {borrowedBooks}");
        Console.WriteLine($"Всего читателей: {_readers.Count}");
        Console.WriteLine($"Общая стоимость книг: {totalValue:C}");
        
        var booksByType = _books.Values
            .GroupBy(b => b.GetType().Name)
            .Select(g => new { Type = g.Key, Count = g.Count() });
        
        Console.WriteLine("\nКниги по типам:");
        foreach (var group in booksByType)
        {
            Console.WriteLine($"  {group.Type}: {group.Count} шт.");
        }
    }
    
    protected virtual void OnBookAdded(Book book)
    {
        BookAdded?.Invoke(this, new BookAddedEventArgs(book));
    }
    
    protected virtual void UpdateStatistics()
    {
        StatisticsUpdated?.Invoke(this, 
            new LibraryStatisticsEventArgs(_books.Count, _readers.Count));
    }
}

// НОВЫЕ КЛАССЫ ДЛЯ АРГУМЕНТОВ СОБЫТИЙ
public class BookAddedEventArgs : EventArgs
{
    public Book Book { get; }
    
    public BookAddedEventArgs(Book book)
    {
        Book = book;
    }
}

public class LibraryStatisticsEventArgs : EventArgs
{
    public int BookCount { get; }
    public int ReaderCount { get; }
    
    public LibraryStatisticsEventArgs(int bookCount, int readerCount)
    {
        BookCount = bookCount;
        ReaderCount = readerCount;
    }
}

// КЛАСС TEXTBOOK 
public class Textbook : Book, IResearchable
{
    private string _subject;
    private string _educationLevel;
    private bool _hasExercises;
    private int _citationCount;
    private List<string> _chapters;

    public Textbook(string title, string author, int yearOfPublication, int pageCount, 
                   string publisher, decimal price, string subject, string educationLevel, bool hasExercises) 
        : base(title, author, yearOfPublication, pageCount, publisher, price)
    {
        Subject = subject;
        EducationLevel = educationLevel;
        HasExercises = hasExercises;
        _citationCount = 0;
        _chapters = new List<string>();
    }

    public string Subject
    {
        get { return _subject; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _subject = value;
            else
                throw new ArgumentException("Предмет не может быть пустым!");
        }
    }

    public string EducationLevel
    {
        get { return _educationLevel; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _educationLevel = value;
            else
                throw new ArgumentException("Уровень образования не может быть пустым!");
        }
    }

    public bool HasExercises
    {
        get { return _hasExercises; }
        set { _hasExercises = value; }
    }

   
    public void AddChapter(string chapter)
    {
        _chapters.Add(chapter);
        Console.WriteLine($"Добавлена глава '{chapter}' в учебник '{Title}'");
    }

   
    public void DisplayChapters()
    {
        Console.WriteLine($"\nГлавы учебника '{Title}':");
        if (_chapters.Any())
        {
            _chapters.ForEach(chapter => Console.WriteLine($"- {chapter}"));
        }
        else
        {
            Console.WriteLine("Нет глав");
        }
    }

    // ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСА IResearchable
    void IResearchable.Research()
    {
        Console.WriteLine($"Проведение исследования по учебнику '{Title}'");
        Console.WriteLine($"Предмет: {Subject}, Уровень: {EducationLevel}");
    }

    int IResearchable.GetCitationCount()
    {
        return _citationCount;
    }

    // Переопределение методов
    public override void Read()
    {
        base.Read();
        Console.WriteLine($"Изучение учебника по предмету '{Subject}' (уровень: {EducationLevel})");
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Предмет: {Subject}, Уровень: {EducationLevel}";
    }

    // Переопределение CalculateValue
    public override decimal CalculateValue()
    {
        decimal baseValue = base.CalculateValue();
        if (EducationLevel == "Высшее образование") baseValue *= 1.3m;
        if (HasExercises) baseValue *= 1.1m;
        return baseValue;
    }

    public void Cite()
    {
        _citationCount++;
        Console.WriteLine($"Учебник '{Title}' процитирован. Всего цитирований: {_citationCount}");
    }
}

// КЛАСС FICTION 
public class Fiction : Book, ILibraryItem
{
    private string _genre;
    private string _literaryPeriod;
    private bool _isBestseller;
    private List<string> _reviews;

    public Fiction(string title, string author, int yearOfPublication, int pageCount, 
                  string publisher, decimal price, string genre, string literaryPeriod, bool isBestseller) 
        : base(title, author, yearOfPublication, pageCount, publisher, price)
    {
        Genre = genre;
        LiteraryPeriod = literaryPeriod;
        IsBestseller = isBestseller;
        _reviews = new List<string>();
    }

    public string Genre
    {
        get { return _genre; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _genre = value;
            else
                throw new ArgumentException("Жанр не может быть пустым!");
        }
    }

    public string LiteraryPeriod
    {
        get { return _literaryPeriod; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _literaryPeriod = value;
            else
                throw new ArgumentException("Литературный период не может быть пустым!");
        }
    }

    public bool IsBestseller
    {
        get { return _isBestseller; }
        set { _isBestseller = value; }
    }

    // МЕТОД для добавления отзывов
    public void AddReview(string review)
    {
        _reviews.Add(review);
        Console.WriteLine($"Добавлен отзыв на книгу '{Title}'");
    }

    // МЕТОД для отображения отзывов
    public void DisplayReviews()
    {
        Console.WriteLine($"\nОтзывы на книгу '{Title}':");
        if (_reviews.Any())
        {
            _reviews.ForEach(review => Console.WriteLine($"- \"{review}\""));
        }
        else
        {
            Console.WriteLine("Нет отзывов");
        }
    }

    // РЕАЛИЗАЦИЯ ДОПОЛНИТЕЛЬНОГО МЕТОДА DisplayInfo
    void ILibraryItem.DisplayInfo()
    {
        Console.WriteLine(GetDetailedInfo());
    }

    // Реализация остальных методов интерфейса ILibraryItem
    string ILibraryItem.Title => Title;
    string ILibraryItem.Author => Author;
    decimal ILibraryItem.CalculateValue() => CalculateValue();

    private string GetDetailedInfo()
    {
        return base.GetInfo() + $", Жанр: {Genre}, Период: {LiteraryPeriod}, " +
               $"Бестселлер: {(IsBestseller ? "Да" : "Нет")}, Отзывов: {_reviews.Count}";
    }

    public override string GetInfo()
    {
        return GetDetailedInfo();
    }

    // Переопределение CalculateValue
    public override decimal CalculateValue()
    {
        decimal baseValue = base.CalculateValue();
        if (IsBestseller) baseValue *= 2m;
        if (LiteraryPeriod == "Классика") baseValue *= 1.5m;
        return baseValue;
    }

    // Переопределение метода для реставрации
    public override void Restore(int conditionPoints)
    {
        base.Restore(conditionPoints);
        Console.WriteLine($"Особый уход за художественной книгой '{Title}'");
    }
}

// НОВЫЙ КЛАСС - Аудиокнига 
public class Audiobook : Book
{
    private double _duration;
    private string _narrator;
    private List<string> _formats;

    // ИЗМЕНЕНО: передаем 0 в pageCount, так как теперь это разрешено
    public Audiobook(string title, string author, int yearOfPublication, 
                    string publisher, decimal price, double duration, string narrator) 
        : base(title, author, yearOfPublication, 0, publisher, price)
    {
        Duration = duration;
        Narrator = narrator;
        _formats = new List<string> { "MP3", "AAC" };
    }

    public double Duration
    {
        get { return _duration; }
        set 
        { 
            if (value > 0)
                _duration = value;
            else
                throw new ArgumentException("Длительность должна быть положительной!");
        }
    }

    public string Narrator
    {
        get { return _narrator; }
        set 
        { 
            if (!string.IsNullOrEmpty(value))
                _narrator = value;
            else
                throw new ArgumentException("Чтец не может быть пустым!");
        }
    }

    // НОВЫЙ МЕТОД для добавления форматов
    public void AddFormat(string format)
    {
        if (!_formats.Contains(format))
        {
            _formats.Add(format);
        }
    }

    //переопределяем метод Read для аудиокниги
    public override void Read()
    {
        ReadCount++;
        Condition = Math.Max(0, Condition - 1);
        
        OnBookRead($"Прослушана аудиокнига '{Title}'", this);
        
        Console.WriteLine($"Прослушивание аудиокниги: '{Title}', Длительность: {Duration}ч, Чтец: {Narrator}");
        Console.WriteLine($"Состояние аудиокниги: {Condition}%");
    }

    // переопределяем GetInfo для аудиокниги
    public override string GetInfo()
    {
        string status = IsBorrowed ? $" (Взята читателем: {CurrentReader?.Name})" : " (Доступна)";
        string tagsInfo = Tags.Any() ? $", Теги: {string.Join(", ", Tags)}" : "";
        return $"Аудиокнига ID:{Id}: '{Title}', Автор: {Author}, Год: {YearOfPublication}, " +
               $"Длительность: {Duration}ч, Чтец: {Narrator}, Издатель: {Publisher}, " +
               $"Состояние: {Condition}%, Форматы: {string.Join(", ", _formats)}{tagsInfo}{status}";
    }

    // Переопределение CalculateValue
    public override decimal CalculateValue()
    {
        decimal baseValue = base.CalculateValue();
        baseValue *= (decimal)(Duration / 10.0);
        return baseValue;
    }
}

// ДЕМОНСТРАЦИЯ РАБОТЫ 
Console.WriteLine("=== ДЕМОНСТРАЦИЯ РАБОТЫ С БИБЛИОТЕКОЙ ===\n");

Library library = new Library();

// Подписка на события библиотеки
library.BookAdded += (sender, args) => 
    Console.WriteLine($"СОБЫТИЕ: Добавлена новая книга в библиотеку: {args.Book.Title}");

library.StatisticsUpdated += (sender, args) =>
    Console.WriteLine($"СОБЫТИЕ: Обновлена статистика - книг: {args.BookCount}, читателей: {args.ReaderCount}");


Reader reader1 = new Reader("Алексей", "alex@mail.com");
Reader reader2 = new Reader("Мария", "maria@mail.com");

library.AddReader(reader1);
library.AddReader(reader2);

// Подписка на события читателей
reader1.BookBorrowed += (sender, args) =>
    Console.WriteLine($"СОБЫТИЕ: {((Reader)sender).Name} взял книгу '{args.Book.Title}'");


Console.WriteLine("Создание книг...");

Textbook mathTextbook = new Textbook("Высшая математика", "Иванов", 2020, 400, "Наука", 1200, 
                                    "Математика", "Высшее образование", true);

Fiction novel = new Fiction("Война и мир", "Толстой", 1869, 1225, "Классика", 800, 
                           "Роман", "Классика", true);

// ИСПРАВЛЕНО: создаем Audiobook с 0 страниц
Audiobook audiobook = new Audiobook("Гарри Поттер", "Роулинг", 2001, "Bloomsbury", 700, 24.5, "Иванов");

Console.WriteLine("Книги успешно созданы!");

// Добавление тегов к книгам
mathTextbook.AddTag("учебник");
mathTextbook.AddTag("математика");

novel.AddTag("классика");
novel.AddTag("роман");

audiobook.AddTag("фэнтези");
audiobook.AddTag("аудиокнига");

// Подписка на события книг
mathTextbook.StatusChanged += (sender, args) =>
    Console.WriteLine($"СОБЫТИЕ: Статус книги '{args.Book.Title}' изменен: {args.Action}");

// Добавление книг в библиотеку
library.AddBook(mathTextbook);
library.AddBook(novel);
library.AddBook(audiobook);

// 
Console.WriteLine("\n=== ТЕСТИРОВАНИЕ ОСНОВНЫХ ФУНКЦИЙ ===");

// Выводим информацию о книгах
Console.WriteLine("\nИнформация о книгах в библиотеке:");
library.ProcessAllBooks(book => {
    Console.WriteLine($"\n{book.GetInfo()}");
    Console.WriteLine($"  Стоимость: {book.CalculateValue():C}");
});

// Выдача книг
Console.WriteLine("\nВыдача книг читателям:");
mathTextbook.Borrow(reader1);
novel.Borrow(reader2);

// Чтение книг
Console.WriteLine("\nЧтение книг:");
mathTextbook.Read();
novel.Read(100);
audiobook.Read();

// Возврат книги
Console.WriteLine("\nВозврат книги:");
mathTextbook.Return();

// Статистика библиотеки
library.DisplayStatistics();

// 
Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ДЕЛЕГАТОВ И ЛЯМБДА-ВЫРАЖЕНИЙ ===");

// Делегат для форматирования
BookFormatDelegate formatDelegate = (book) => 
    $"{book.Title} - {book.Author} ({book.YearOfPublication}) - {book.CalculateValue():C}";

Console.WriteLine("\nФорматированная информация о книгах:");
foreach (var book in new List<Book> { mathTextbook, novel, audiobook })
{
    Console.WriteLine($"  {book.FormatInfo(formatDelegate)}");
}

// Фильтрация с использованием лямбда-выражений
Console.WriteLine("\nДорогие книги (стоимость > 1000):");
var expensiveBooks = library.FindBooks(b => b.CalculateValue() > 1000);
foreach (var book in expensiveBooks)
{
    Console.WriteLine($"  - {book.Title}: {book.CalculateValue():C}");
}

Console.WriteLine("\n=== РАБОТА С КОЛЛЕКЦИЯМИ ===");

// История заимствований читателя
reader1.DisplayBorrowHistory();

Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА ===");

=== ДЕМОНСТРАЦИЯ РАБОТЫ С БИБЛИОТЕКОЙ ===

СОБЫТИЕ: Обновлена статистика - книг: 0, читателей: 1
СОБЫТИЕ: Обновлена статистика - книг: 0, читателей: 2
Создание книг...
Книги успешно созданы!
Добавлен тег 'учебник' к книге 'Высшая математика'
Добавлен тег 'математика' к книге 'Высшая математика'
Добавлен тег 'классика' к книге 'Война и мир'
Добавлен тег 'роман' к книге 'Война и мир'
Добавлен тег 'фэнтези' к книге 'Гарри Поттер'
Добавлен тег 'аудиокнига' к книге 'Гарри Поттер'
СОБЫТИЕ: Добавлена новая книга в библиотеку: Высшая математика
СОБЫТИЕ: Обновлена статистика - книг: 1, читателей: 2
СОБЫТИЕ: Добавлена новая книга в библиотеку: Война и мир
СОБЫТИЕ: Обновлена статистика - книг: 2, читателей: 2
СОБЫТИЕ: Добавлена новая книга в библиотеку: Гарри Поттер
СОБЫТИЕ: Обновлена статистика - книг: 3, читателей: 2

=== ТЕСТИРОВАНИЕ ОСНОВНЫХ ФУНКЦИЙ ===

Информация о книгах в библиотеке:

Обработка всех книг в библиотеке (3 шт.):

Книга ID:1: 'Высшая математика', Автор: Иванов, Год: 2020, Стр